# 🔬 MNIST LAB: Śledztwo w sprawie najważniejszych pikseli
### Od `RandomForest.feature_importances_` do eksperymentów, które naprawdę testują hipotezy

**Laboratorium na szkolenie • Python + scikit-learn • 28 × 28 = 784 cechy • 10 klas**

Punktem wyjścia jest załączony notebook z **Olivetti Faces**: `RandomForestClassifier → feature_importances_ → reshape → heatmapa`. Tutaj zachowujemy ten prosty, genialny pomysł i rozbudowujemy go o **siedem perspektyw na ważność**.

> **Pytanie przewodnie:** Czy „ważny piksel” jest ważny zawsze, dla każdej cyfry i niezależnie od sposobu mierzenia ważności?

| Etap | Eksperyment | Czego się dowiemy? |
|:--|:--|:--|
| 01 | Galeria i statystyka danych | Co naprawdę widzi algorytm? |
| 02 | Las losowy i MDI | Jak model drzew ocenia piksele? |
| 03 | Informacja wzajemna (MI) | Co mówi nam zależność z etykietą, bez trenowania lasu? |
| 04 | Liniowy model SGD, mapy podpisane | Które piksele **wspierają** rozpoznanie konkretnej cyfry, a które mu **przeczą**? |
| 05 | Zestawienie trzech metod | Dlaczego różne definicje „ważności” dają inne odpowiedzi? |
| 06 | Permutacja **grup pikseli** | Co się stanie, gdy zniszczymy informację w fragmentach obrazu? |
| 07 | Ablacja i selekcja top-*k* | Co się stanie, gdy usuniemy lub zachowamy wybrane piksele? |
| 08 | Niewidziany zbiór testowy, błędy | Czy efekt utrzymuje się poza danymi treningowymi? |
| 09 | Stabilność, eksport | Czy ranking jest powtarzalny i jak zapisać wyniki? |

**Uruchamianie:** `Run → Run All`. Domyślny `FAST_MODE=True` jest skonfigurowany do prowadzenia zajęć. Przy pierwszym uruchomieniu **wymagany internet** do pobrania prawdziwego zbioru MNIST (70 000 obrazów). Notebook **nie zastępuje MNIST zbiorem `load_digits`**. Potrzebne pakiety: `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn` (zalecane Python ≥ 3.10).

<small>Metody MDI, MI, wagi liniowe, permutacje i ablacje mierzą różne wielkości. Wynik nie jest uniwersalną ani przyczynową „prawdą” o pikselach.</small>

## 00 · Ustawienia laboratorium

- `FAST_MODE=True`: około 12 tys. obrazów treningowych, mniej drzew i mniejsza liczba permutacji.
- `FAST_MODE=False`: większa próba, dokładniejsze mapy, dłuższe obliczenia.
- `DATA_SOURCE='auto'`: próba `fetch_openml`, później `torchvision` lub `tensorflow.keras`; każda metoda pobiera **rzeczywisty MNIST**.
- **Trzy rozłączne zbiory:** trening do uczenia i rankingu, walidacja do eksperymentów i wyboru *k*, test do jednorazowej kontroli końcowej.

Komórka instalacyjna (uruchom tylko w razie brakujących pakietów):

```python
# %pip install -U numpy pandas matplotlib scipy scikit-learn
```

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import spearmanr

from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
FAST_MODE = True
DATA_SOURCE = 'auto'       # auto / openml / torchvision / keras
N_JOBS = -1
TRAIN_LIMIT = 12_000 if FAST_MODE else 50_000
VALID_LIMIT = 2_500 if FAST_MODE else 10_000
TEST_LIMIT = 3_000 if FAST_MODE else 10_000
N_TREES = 140 if FAST_MODE else 280
MI_SAMPLE = 3_500 if FAST_MODE else 10_000
PATCH_SIZE = 7 if FAST_MODE else 4
PATCH_REPEATS = 2 if FAST_MODE else 3
RUN_STABILITY = False   # dodatkowy eksperyment, świadomie wyłączony w trybie szkoleniowym
OUT_DIR = Path('mnist_piksele_wyniki')
OUT_DIR.mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)

plt.rcParams.update({
    'figure.facecolor': '#091625', 'axes.facecolor': '#10243a',
    'savefig.facecolor': '#091625', 'text.color': '#f0f5fb',
    'axes.labelcolor': '#e3edf8', 'axes.edgecolor': '#8ca6bf',
    'xtick.color': '#a6c0d5', 'ytick.color': '#a6c0d5',
    'font.size': 11, 'axes.titlesize': 13, 'figure.titlesize': 18,
    'grid.alpha': .20,
})

def savefig(filename):
    plt.savefig(OUT_DIR / filename, dpi=170, bbox_inches='tight')

def show_heat(ax, scores, title, *, cmap='magma', symmetric=False):
    img = np.asarray(scores).reshape(28, 28)
    if symmetric:
        vmax = max(float(np.max(np.abs(img))), 1e-9)
        im = ax.imshow(img, cmap=cmap, interpolation='nearest', vmin=-vmax, vmax=vmax)
    else:
        im = ax.imshow(img, cmap=cmap, interpolation='nearest')
    ax.set_title(title, pad=12)
    ax.set_xticks([]); ax.set_yticks([])
    return im

print(f'MNIST LAB | FAST_MODE={FAST_MODE} | drzewa={N_TREES} | źródło={DATA_SOURCE}')

## 01 · Pobranie prawdziwego MNIST i szczelny podział danych

Każdy obraz ma **28 × 28** poziomów szarości (0–255). Rozwijamy macierz obrazu do wektora 784 cech. Pierwsze pobranie może potrwać, kolejne korzysta z pamięci podręcznej użytej biblioteki.

**Ważne dydaktycznie:** ranking cech powstaje **tylko z treningu**. Nie podglądamy zbioru testowego przy wyborze metody ani liczby pikseli.

In [ ]:
def load_real_mnist(source='auto'):
    errors = []
    sources = ['openml', 'torchvision', 'keras'] if source == 'auto' else [source]
    for s in sources:
        try:
            if s == 'openml':
                print('Pobieram MNIST z OpenML / cache scikit-learn...')
                try:
                    ds = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
                except TypeError:  # starsze wydania scikit-learn
                    ds = fetch_openml('mnist_784', version=1, as_frame=False)
                X, y = np.asarray(ds.data), np.asarray(ds.target)
            elif s == 'torchvision':
                print('Pobieram MNIST przez torchvision...')
                from torchvision.datasets import MNIST
                root = str(Path.home() / '.cache' / 'mnist_lab')
                tr = MNIST(root=root, train=True, download=True)
                te = MNIST(root=root, train=False, download=True)
                X = np.concatenate([np.asarray(tr.data), np.asarray(te.data)], axis=0)
                y = np.concatenate([np.asarray(tr.targets), np.asarray(te.targets)])
            elif s == 'keras':
                print('Pobieram MNIST przez tensorflow.keras...')
                from tensorflow.keras.datasets import mnist
                (xt, yt), (xe, ye) = mnist.load_data()
                X = np.concatenate([xt, xe], axis=0)
                y = np.concatenate([yt, ye])
            else:
                raise ValueError('DATA_SOURCE musi być auto / openml / torchvision / keras')
            X = np.asarray(X, dtype=np.uint8).reshape(-1, 784)
            y = np.asarray(y, dtype=np.int64).reshape(-1)
            assert X.shape == (70_000, 784) and y.shape == (70_000,)
            assert np.array_equal(np.unique(y), np.arange(10))
            print(f'Sukces: {s}, dane = {X.shape}, zakres = [{X.min()}, {X.max()}]')
            return X, y
        except Exception as exc:
            errors.append(f'{s}: {type(exc).__name__}: {str(exc)[:180]}')
            print(f'Źródło {s} niedostępne: {type(exc).__name__}')
    raise RuntimeError(
        'Nie udało się pobrać PRAWDZIWEGO MNIST. Sprawdź internet lub zmień DATA_SOURCE.\n'
        + '\n'.join(errors)
    )

def stratified_cap(X, y, n, seed):
    if n is None or n >= len(y):
        return X, y
    idx, _ = train_test_split(np.arange(len(y)), train_size=n,
                              stratify=y, random_state=seed)
    return X[idx], y[idx]

X_all, y_all = load_real_mnist(DATA_SOURCE)
X_develop, X_test, y_develop, y_test = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_develop, y_develop, test_size=(0.15 / 0.85),
    stratify=y_develop, random_state=SEED)
X_train, y_train = stratified_cap(X_train, y_train, TRAIN_LIMIT, SEED + 1)
X_val, y_val = stratified_cap(X_val, y_val, VALID_LIMIT, SEED + 2)
X_test, y_test = stratified_cap(X_test, y_test, TEST_LIMIT, SEED + 3)
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1] == 784
assert len(set(y_train)) == len(set(y_val)) == len(set(y_test)) == 10
print(pd.DataFrame({'zbiór':['Trening', 'Walidacja', 'Test'],
                    'obrazów':[len(y_train), len(y_val), len(y_test)],
                    'cech':[784]*3}).to_string(index=False))

## 02 · Galeria cyfr: skąd model czerpie informację?

**Paradoks:** ludzkie oko widzi „ósemkę”, a algorytm widzi wyłącznie 784 liczb. Zobaczmy przykłady, uśrednioną sylwetkę każdej cyfry i zmienność intensywności pikseli.

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(16, 7))
for digit in range(10):
    ix = np.flatnonzero(y_train == digit)
    chosen = RNG.choice(ix, size=4, replace=False)
    for row, idx in enumerate(chosen):
        axes[row, digit].imshow(X_train[idx].reshape(28, 28), cmap='gray', vmin=0, vmax=255)
        axes[row, digit].axis('off')
        if row == 0:
            axes[row, digit].set_title(str(digit), fontsize=16, color='#53dbfa')
fig.suptitle('40 obrazów · 10 klas · żadnych ręcznie zdefiniowanych reguł', y=1.01)
plt.tight_layout(); savefig('01_galeria.png'); plt.show()

mean_digits = np.stack([X_train[y_train == d].mean(axis=0) for d in range(10)])
fig, axes = plt.subplots(2, 5, figsize=(13, 5.7))
for d, ax in enumerate(axes.flat):
    show_heat(ax, mean_digits[d], f'Średnia cyfra {d}', cmap='gray')
fig.suptitle('Średnie cyfry: struktura probabilistyczna, nie szablon wzorcowy')
plt.tight_layout(); savefig('02_srednie_cyfry.png'); plt.show()

## 03 · Najprostsza mapa: aktywność i wariancja

Piksel, który nie zmienia się w zbiorze, nie pomaga rozróżniać klas. Ale **duża wariancja nie oznacza automatycznie dużej wartości dla klasyfikacji**. Porównajmy statystyki bez etykiet z ważnością nauczonego modelu (w następnym etapie).

In [ ]:
pixel_mean = X_train.mean(axis=0)
pixel_var = X_train.astype(np.float32).var(axis=0)
fig, ax = plt.subplots(1, 2, figsize=(11.8, 5.0))
im0 = show_heat(ax[0], pixel_mean, 'Średnia intensywność')
im1 = show_heat(ax[1], pixel_var, 'Wariancja intensywności')
fig.colorbar(im0, ax=ax[0], fraction=.045, pad=.03)
fig.colorbar(im1, ax=ax[1], fraction=.045, pad=.03)
fig.suptitle('Informacja o obrazach ≠ informacja o etykietach')
plt.tight_layout(); savefig('03_statystyki.png'); plt.show()

## 04 · Random Forest: importances przez spadek nieczystości (MDI)

Reprodukujemy rdzeń oryginalnego notebooku: **las losowy → `feature_importances_` → `reshape(28, 28)` → mapa `hot`**. Las w każdym drzewie dzieli przestrzeń na obszary, a MDI sumuje ważone redukcje nieczystości w węzłach korzystających z danej cechy.

**Ograniczenie:** MDI może dzielić znaczenie między piksele skorelowane oraz faworyzować cechy oferujące więcej możliwych punktów podziału. To ważność *wewnętrzna* lasu, nie uniwersalna miara prawdy.

In [ ]:
t0 = time.perf_counter()
forest = RandomForestClassifier(
    n_estimators=N_TREES, max_features='sqrt', min_samples_leaf=1,
    n_jobs=N_JOBS, random_state=SEED)
forest.fit(X_train, y_train)
mdi = forest.feature_importances_
val_pred_forest = forest.predict(X_val)
val_acc_forest = accuracy_score(y_val, val_pred_forest)
print(f'Las: fit + prognoza = {time.perf_counter()-t0:.1f} s')
print(f'Accuracy na WALIDACJI = {val_acc_forest:.4f}')
print(f'Suma MDI = {mdi.sum():.6f} ; największa wartość = {mdi.max():.6f}')

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
im = show_heat(ax[0], mdi, 'Ważność MDI · Random Forest', cmap='hot')
ax[1].imshow(X_val[0].reshape(28, 28), cmap='gray', vmin=0, vmax=255)
ax[1].imshow(mdi.reshape(28,28), cmap='hot', alpha=.60)
ax[1].set_title(f'Nakładka na prawdziwą cyfrę {y_val[0]}')
ax[1].axis('off')
fig.colorbar(im, ax=ax[0], fraction=.045, pad=.03)
fig.suptitle('Punkt wyjścia z Olivetti: te same 784 piksele, nowe znaczenie')
plt.tight_layout(); savefig('04_mdi_random_forest.png'); plt.show()

## 05 · Gdzie dokładnie znajdują się TOP-50 pikseli?

Wynik zapisujemy z **numerem piksela**, współrzędnymi **(wiersz, kolumna)** i wagą. Możesz pokazać uczestnikom, że algorytm znajduje nie „cechę: oczko w ósemce”, lecz konkretne pozycje na siatce.

In [ ]:
def ranking_table(scores, n=20):
    order = np.argsort(np.asarray(scores))[::-1][:n]
    return pd.DataFrame({
        'ranking': np.arange(1, len(order)+1), 'piksel': order,
        'wiersz': order // 28, 'kolumna': order % 28,
        'ważność': np.asarray(scores)[order]})

print('TOP-20 wg MDI')
display(ranking_table(mdi, 20))
top50 = np.argsort(mdi)[-50:]
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(mdi.reshape(28,28), cmap='magma')
axes[0].scatter(top50 % 28, top50 // 28, s=25, facecolors='none',
                edgecolors='#49e3ff', linewidths=1.2)
axes[0].set_title('Pierścienie: 50 najważniejszych pikseli')
axes[1].imshow(mean_digits[8].reshape(28,28), cmap='gray')
axes[1].scatter(top50 % 28, top50 // 28, s=25, facecolors='none',
                edgecolors='#ffcf5a', linewidths=1.3)
axes[1].set_title('Te same pozycje na średniej cyfrze 8')
for a in axes: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); savefig('05_top50_mdi.png'); plt.show()

## 06 · Mutual Information: które piksele niosą informację o etykiecie?

**Informacja wzajemna** $I(X_j;Y)$ mierzy statystyczną zależność między intensywnością piksela a klasą (bez trenowania lasu). Jest nieujemna i nie opisuje kierunku relacji. Dla szybkości i stabilności kwantyzujemy jasności do **16 koszyków**; jest to nowy wybór analityczny, nie cecha oryginalnego zbioru.

> Nie należy porównywać bezpośrednio liczbowych wartości MI i MDI: różnią się jednostkami i definicją. Porównujemy przestrzenne wzory, rankingi i części wspólne zbiorów TOP-*k*.

In [ ]:
X_mi, y_mi = stratified_cap(X_train, y_train, MI_SAMPLE, SEED + 10)
X_mi_bins = (X_mi // 16).astype(np.uint8)
t0 = time.perf_counter()
mi = mutual_info_classif(X_mi_bins, y_mi, discrete_features=True,
                         random_state=SEED)
print(f'MI gotowa po {time.perf_counter()-t0:.1f}s na {len(y_mi)} obrazach')
fig, axes = plt.subplots(1, 3, figsize=(15.2, 4.6))
show_heat(axes[0], mdi, 'Random Forest · MDI', cmap='hot')
show_heat(axes[1], mi, 'Mutual Information · MI', cmap='viridis')
show_heat(axes[2], pixel_var, 'Wariancja · bez etykiet', cmap='magma')
fig.suptitle('Trzy mapy, trzy pytania')
plt.tight_layout(); savefig('06_mdi_mi_wariancja.png'); plt.show()

## 07 · Mapy dla **konkretnej cyfry**: model liniowy

`SGDClassifier(loss='log_loss')` uczy liniowy klasyfikator logistyczny z zestawem wag osobnym dla każdej cyfry. **Waga dodatnia** wspiera wynik danej klasy, **ujemna** obniża go, gdy intensywność piksela rośnie, *przy pozostałych cechach stałych*. Przy silnych korelacjach między pikselami współczynniki **nie są wyjaśnieniem przyczynowym**.

Standaryzację dopasowujemy na samym treningu. Aby mapy były w jednostkach wejściowych, dzielimy `coef_` przez `scale_` transformacji.

In [ ]:
def make_linear_model(seed=SEED):
    return make_pipeline(
        StandardScaler(),
        SGDClassifier(loss='log_loss', alpha=0.001, penalty='l2',
                      max_iter=1000, tol=1e-3, random_state=seed,
                      average=True))

t0 = time.perf_counter()
linear = make_linear_model()
linear.fit(X_train, y_train)
linear_val_acc = accuracy_score(y_val, linear.predict(X_val))
scaler = linear.named_steps['standardscaler']
sgd = linear.named_steps['sgdclassifier']
class_weights = sgd.coef_ / scaler.scale_[None, :]
abs_linear = np.mean(np.abs(class_weights), axis=0)
print(f'SGD logistyczny: czas = {time.perf_counter()-t0:.1f}s | walidacja = {linear_val_acc:.4f}')
print('Macierz wag (klasa × piksel):', class_weights.shape)

fig, ax = plt.subplots(2, 5, figsize=(14, 6))
for d, a in enumerate(ax.flat):
    show_heat(a, class_weights[d], f'Cyfra {d}: + wspiera, − przeczy',
              cmap='coolwarm', symmetric=True)
fig.suptitle('10 podpisanych map ważności · 10 różnych decyzji', y=1.01)
plt.tight_layout(); savefig('07_dziesiec_map_podpisanych.png'); plt.show()

## 08 · Pod mikroskopem: cyfra 3 kontra cyfra 8

Dla każdej klasy rysujemy średni obraz oraz wagę liniową. **Błękit i czerwień** to nie „dobry i zły” piksel, lecz znak wpływu piksela na wynik klasy.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12.5, 8))
for r, d in enumerate([3, 8]):
    show_heat(ax[r, 0], mean_digits[d], f'Średnia cyfra {d}', cmap='gray')
    show_heat(ax[r, 1], class_weights[d], f'Wagi klasy {d}', cmap='coolwarm', symmetric=True)
    a = ax[r, 2]
    a.imshow(mean_digits[d].reshape(28,28), cmap='gray', vmin=0, vmax=255)
    weight = class_weights[d].reshape(28,28)
    lim = np.max(np.abs(weight)) + 1e-12
    a.imshow(weight, cmap='coolwarm', norm=TwoSlopeNorm(vcenter=0, vmin=-lim, vmax=lim), alpha=.57)
    a.set_title(f'Nakładka: ślad decyzji dla {d}'); a.axis('off')
fig.suptitle('Znaczenie piksela zależy od pytania: „jaka cyfra?”', y=1.01)
plt.tight_layout(); savefig('08_3_kontra_8.png'); plt.show()

## 09 · Spór metod: MDI vs MI vs średnia bezwzględna waga liniowa

Rankingiem może być porządek cech, niekoniecznie wspólna skala liczbowa. Obliczymy **korelację rang Spearmana** oraz **Jaccarda** dla zbiorów TOP-50. Nawet duża zbieżność nie oznacza, że uzyskaliśmy „prawdziwe” znaczenie przyczynowe.

In [ ]:
methods = {'MDI (forest)': mdi, 'MI (etykieta)': mi, '|waga| (linear)': abs_linear}
method_names = list(methods)
rank_corr = pd.DataFrame(index=method_names, columns=method_names, dtype=float)
intersection = pd.DataFrame(index=method_names, columns=method_names, dtype=float)

def top_set(v, k=50):
    return set(np.argsort(v)[-k:].tolist())

for a in method_names:
    for b in method_names:
        rank_corr.loc[a,b] = spearmanr(methods[a], methods[b]).statistic
        A, B = top_set(methods[a]), top_set(methods[b])
        intersection.loc[a,b] = len(A & B) / len(A | B)

fig, ax = plt.subplots(1, 2, figsize=(13.5, 5.1))
for a, (data, title) in zip(ax, [(rank_corr,'Korelacja rang Spearmana'),
                                (intersection,'Jaccard TOP-50')]):
    im = a.imshow(data.astype(float), vmin=0, vmax=1, cmap='viridis')
    a.set_xticks(range(3), method_names, rotation=30, ha='right')
    a.set_yticks(range(3), method_names)
    for i in range(3):
        for j in range(3):
            a.text(j, i, f'{data.iloc[i,j]:.2f}', ha='center', va='center',
                   color='white' if data.iloc[i,j] < .65 else '#071423', fontsize=12)
    a.set_title(title)
fig.suptitle('Zgodność metod to wynik eksperymentu, nie założenie')
plt.tight_layout(); savefig('09_zgodnosc_metod.png'); plt.show()
print('Korelacje rang:'); display(rank_corr.round(3))
print('Jaccard TOP-50:'); display(intersection.round(3))

## 10 · Permutation importance, ale w **grupach**: zasłanianie informacji

Standardowe permutowanie każdego z 784 pikseli osobno jest wolne i ignoruje lokalną strukturę. Stosujemy więc **grupową permutację bloków** obrazu. Wszystkie piksele w danym bloku zamieniamy *razem między przykładami*, co zachowuje strukturę wewnątrz bloku, ale odrywa ją od poprawnej etykiety.

Dla każdego bloku: $\Delta \mathrm{Acc} = \mathrm{Acc}_{bazowa} - \mathrm{Acc}_{po\ permutacji}$. Wartości mogą być **ujemne**. Używamy *walidacji*, nie testu. Ten eksperyment mierzy ważność **bloku dla danego, wytrenowanego lasu**.

In [ ]:
def patch_permutation_importance(model, X, y, patch_size=7, repeats=2, seed=42):
    assert 28 % patch_size == 0, 'Rozmiar płytki musi dzielić 28'
    X = np.asarray(X)
    rng = np.random.default_rng(seed)
    base = accuracy_score(y, model.predict(X))
    nblocks = 28 // patch_size
    drops = np.zeros((nblocks, nblocks), dtype=float)
    for rr in range(nblocks):
        for cc in range(nblocks):
            pix = np.array([(r * 28 + c)
                            for r in range(rr * patch_size, (rr+1)*patch_size)
                            for c in range(cc * patch_size, (cc+1)*patch_size)])
            current = []
            for _ in range(repeats):
                perm = rng.permutation(len(y))
                corrupted = X.copy()
                corrupted[:, pix] = X[perm][:, pix]
                acc = accuracy_score(y, model.predict(corrupted))
                current.append(base - acc)
            drops[rr, cc] = np.mean(current)
    return base, drops

perm_X, perm_y = stratified_cap(X_val, y_val, 550 if FAST_MODE else 1_200, SEED+20)
t0 = time.perf_counter()
base_patch, patch_drops = patch_permutation_importance(
    forest, perm_X, perm_y, patch_size=PATCH_SIZE,
    repeats=PATCH_REPEATS, seed=SEED)
print(f'Grupowa permutacja: {time.perf_counter()-t0:.1f}s | punkt odniesienia = {base_patch:.4f}')

patch_28 = np.kron(patch_drops, np.ones((PATCH_SIZE, PATCH_SIZE)))
fig, ax = plt.subplots(1, 3, figsize=(14.5, 4.8))
show_heat(ax[0], mdi, 'MDI: pojedynczy piksel', cmap='hot')
lim = max(float(np.max(np.abs(patch_drops))), 1e-9)
im = ax[1].imshow(patch_drops, cmap='coolwarm', vmin=-lim, vmax=lim)
ax[1].set_title(f'Zmiana accuracy · bloki {PATCH_SIZE}×{PATCH_SIZE}')
ax[1].set_xticks(range(len(patch_drops)))
ax[1].set_yticks(range(len(patch_drops)))
fig.colorbar(im, ax=ax[1], fraction=.045, pad=.03)
show_heat(ax[2], patch_28, 'Te same bloki na siatce 28×28',
          cmap='coolwarm', symmetric=True)
fig.suptitle('Czasem to nie pojedynczy piksel, lecz fragment pisma')
plt.tight_layout(); savefig('10_permutacje_blokow.png'); plt.show()

## 11 · Ablacja: wymazujemy najważniejsze piksele **bez ponownego treningu**

Trzy scenariusze dla tego samego już wytrenowanego lasu:

1. **TOP MDI:** zamieniamy na czarne (`0`) *k* najwyżej ocenionych pikseli.
2. **BOTTOM MDI:** usuwamy *k* najniżej ocenionych pikseli.
3. **RANDOM:** usuwamy losowe *k* pikseli (średnia z kilku losowań).

**Pytanie:** Czy usunięcie TOP-50 boli bardziej niż usunięcie 50 losowych? **Zastrzeżenie:** wymazywanie tworzy obrazy, których model niekoniecznie widział podczas treningu (*out of distribution*). To test zachowania konkretnego modelu po ingerencji, nie dowód przyczynowy o obrazach jako takich.

In [ ]:
def zero_pixels(X, selected_idx):
    arr = np.array(X, copy=True)
    arr[:, np.asarray(selected_idx, dtype=int)] = 0
    return arr

ablation_X, ablation_y = stratified_cap(X_val, y_val, 900 if FAST_MODE else 2000, SEED + 30)
order = np.argsort(mdi)[::-1]
ks_ablation = [0, 20, 50, 100, 200, 350]
curves = []
for k in ks_ablation:
    top = order[:k]
    bottom = order[-k:] if k else np.array([], dtype=int)
    acc_top = accuracy_score(ablation_y, forest.predict(zero_pixels(ablation_X, top)))
    acc_bottom = accuracy_score(ablation_y, forest.predict(zero_pixels(ablation_X, bottom)))
    rand_acc = []
    for rep in range(3):
        rand_idx = np.random.default_rng(SEED+rep+k).choice(784, k, replace=False)
        rand_acc.append(accuracy_score(ablation_y, forest.predict(zero_pixels(ablation_X, rand_idx))))
    curves.append({'k':k, 'TOP MDI':acc_top, 'BOTTOM MDI':acc_bottom,
                   'RANDOM':float(np.mean(rand_acc))})
curve_df = pd.DataFrame(curves)
fig, ax = plt.subplots(figsize=(10.7, 5.3))
for lab, color, marker in [('TOP MDI','#ff6b6b','o'),
                           ('RANDOM','#48d7fb','s'),('BOTTOM MDI','#fbd45b','^')]:
    ax.plot(curve_df['k'], curve_df[lab], '-'+marker, linewidth=2.7,
            markersize=8, color=color, label=lab)
ax.set(xlabel='Liczba wymazanych pikseli', ylabel='Accuracy na walidacji',
       title='Co naprawdę stanie się z lasem po utracie informacji?')
ax.grid(True); ax.legend()
plt.tight_layout(); savefig('11_ablacja_krzywe.png'); plt.show()
display(curve_df.round(4))

## 12 · „Operacja na cyfrze”: oryginał, usunięcie TOP-100 i RANDOM-100

Jedna wizualizacja często bardziej przemawia do uczestników niż kilkadziesiąt tabel. Wynik modelu podajemy pod każdym obrazem, ale pamiętamy, że interpretujemy **jego reakcję na zmianę danych**.

In [ ]:
examples = [np.flatnonzero(y_val == d)[0] for d in [0, 3, 5, 8]]
random100 = np.random.default_rng(SEED+80).choice(784, 100, replace=False)
fig, ax = plt.subplots(4, 3, figsize=(10, 11))
for row, idx in enumerate(examples):
    img = X_val[idx:idx+1]
    variants = [img, zero_pixels(img, order[:100]), zero_pixels(img, random100)]
    captions = ['Oryginał', 'Bez TOP-100 MDI', 'Bez RANDOM-100']
    for col, (v, cap) in enumerate(zip(variants, captions)):
        pred = forest.predict(v)[0]
        ax[row,col].imshow(v[0].reshape(28,28), cmap='gray', vmin=0, vmax=255)
        ax[row,col].set_title(f'{cap} | prawda {y_val[idx]} → model {pred}',
                              color='#8ff0bd' if pred == y_val[idx] else '#ff8686', fontsize=11)
        ax[row,col].axis('off')
fig.suptitle('Przypadki indywidualne: świetny moment na dyskusję', y=1.005)
plt.tight_layout(); savefig('12_operacja_na_cyfrze.png'); plt.show()

## 13 · Selekcja cech: jak mało pikseli wystarczy do trenowania **nowego** modelu?

Inny eksperyment niż ablacja: zamiast kaleczyć obraz wejściowy **trenujemy od nowa** klasyfikator logistyczny, korzystając wyłącznie z TOP-*k* cech według **MI liczonego na treningu**. Wybór *k* następuje wyłącznie na **walidacji**, nie na teście.

**Porównanie jest uczciwe:** każdy wariant ma tę samą procedurę `StandardScaler → SGDClassifier` i tę samą próbę danych. Wynik dla 784 pikseli też wyliczamy w tym samym eksperymencie. To **nie** jest analiza ablacji lasu, lecz zupełnie inny problem badawczy.

In [ ]:
mi_order = np.argsort(mi)[::-1]
k_grid = [20, 50, 100, 200, 400, 784]
selection_results = []
models_by_k = {}
for k in k_grid:
    selected = mi_order[:k]
    clf = make_linear_model()
    clf.fit(X_train[:, selected], y_train)
    val_accuracy = accuracy_score(y_val, clf.predict(X_val[:, selected]))
    selection_results.append({'piksele':k, 'ułamek_cech':k/784,
                              'accuracy_walidacja':val_accuracy})
    models_by_k[k] = clf
selection_df = pd.DataFrame(selection_results)
best_k = int(selection_df.loc[selection_df['accuracy_walidacja'].idxmax(),'piksele'])
print(f'Wybór WYŁĄCZNIE z walidacji: k={best_k}')

fig, ax = plt.subplots(figsize=(10.2, 5.3))
ax.plot(selection_df['piksele'], selection_df['accuracy_walidacja'], '-o',
        lw=2.8, color='#58dcf9', markersize=8)
ax.axvline(best_k, ls='--', color='#ffce66', lw=1.5, label=f'wybrane k={best_k}')
ax.set(xlabel='Piksele dostępne podczas treningu',
       ylabel='Accuracy na WALIDACJI',
       title='Czy 784 piksele są niezbędne? Testujemy hipotezę')
ax.set_xticks(k_grid)
ax.grid(True); ax.legend()
plt.tight_layout(); savefig('13_wybor_k.png'); plt.show()
display(selection_df.round(4))

## 14 · Jednorazowe otwarcie koperty TEST 🔐

Teraz znamy wynik walidacyjny, mamy wybrany wariant TOP-*k*. Dopiero teraz badamy **niewidziany test** i porównujemy trzy sposoby przewidywania: Random Forest z pełnym obrazem, model liniowy z pełnym obrazem, model liniowy z wybranymi pikselami. Nigdy nie zmieniamy wyboru *k* w reakcji na wynik testowy.

In [ ]:
final_predictions = {
    'Random Forest · 784': forest.predict(X_test),
    'SGD liniowy · 784': models_by_k[784].predict(X_test[:,mi_order[:784]]),
    f'SGD liniowy · TOP-{best_k}': models_by_k[best_k].predict(X_test[:,mi_order[:best_k]])
}
final_scores = pd.DataFrame([
    {'model':name, 'test_accuracy':accuracy_score(y_test, pred)}
    for name,pred in final_predictions.items()]).sort_values('test_accuracy', ascending=False)
print('WYNIKI KOŃCOWE NA TEST (bez dalszego strojenia):')
display(final_scores.round(4))
print('\nRaport klasyfikacji: Random Forest')
print(classification_report(y_test, final_predictions['Random Forest · 784'], digits=3))

cm = confusion_matrix(y_test, final_predictions['Random Forest · 784'], labels=range(10))
fig, ax = plt.subplots(figsize=(9, 8))
disp = ConfusionMatrixDisplay(cm, display_labels=range(10))
disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
ax.set_title('Random Forest · macierz pomyłek na TEST')
plt.tight_layout(); savefig('14_macierz_pomylek_test.png'); plt.show()

## 15 · Autopsja pomyłek: które cyfry wprowadzają model w błąd?

Wyświetlimy kilkanaście błędów wraz z **pewnością lasu** (największą wartością `predict_proba`, niekoniecznie dobrze skalibrowanym prawdopodobieństwem). To kluczowa różnica pomiędzy klasyfikatorem, który umie rozpoznać obraz, i klasyfikatorem, który *wie, kiedy może się mylić*.

In [ ]:
rf_pred = final_predictions['Random Forest · 784']
wrong_ids = np.flatnonzero(rf_pred != y_test)
print(f'Liczba błędów: {len(wrong_ids)} / {len(y_test)}')
chosen_errors = wrong_ids[:min(15, len(wrong_ids))]
if len(chosen_errors):
    prob = forest.predict_proba(X_test[chosen_errors])
    fig, axes = plt.subplots(3, 5, figsize=(13, 8))
    for ax in axes.flat: ax.axis('off')
    for j, (idx, ax) in enumerate(zip(chosen_errors, axes.flat)):
        ax.imshow(X_test[idx].reshape(28,28), cmap='gray', vmin=0, vmax=255)
        ax.set_title(f'prawda {y_test[idx]} → {rf_pred[idx]} | max p={prob[j].max():.2f}',
                     color='#ff9b9b', fontsize=10)
    fig.suptitle('Błędy uczą więcej niż średnia accuracy')
    plt.tight_layout(); savefig('15_bledy_modelu.png'); plt.show()
else:
    print('Brak błędów w ocenianej próbce.')

## 16 · Lokalny mikroskop: dlaczego **ten jeden obraz** został uznany za cyfrę 3?

Dotychczasowe mapy były globalne. Teraz bierzemy **jedną cyfrę**, przesuwamy po niej czarne okienko **4×4**, a po każdej zmianie mierzymy spadek `predict_proba` dla **pierwotnie przewidywanej klasy**:

$\Delta p = p_{model}(\hat y\mid x) - p_{model}(\hat y\mid x_{zasłonięty\ blok})$

- **Ciepłe pola:** zasłonięcie bloku osłabia pierwotne rozpoznanie.
- **Zimne pola:** zasłonięcie bloku wzmacnia pierwotne rozpoznanie.
- **Białe/neutralne:** niemal brak zmiany.

**To lokalna wrażliwość modelu, a nie niezależny, przyczynowy wpływ pikseli.** Blokowane obrazy mogą być poza rozkładem treningowym, a `predict_proba` lasu nie musi być idealnie skalibrowane. Wszystkie przykłady czerpiemy z walidacji.

In [ ]:
def local_occlusion(model, image, patch_size=4):
    """Wektorowo testuj spadek wyniku klasy po zerowaniu kolejnych bloków."""
    assert 28 % patch_size == 0
    image = np.asarray(image).reshape(1, 784)
    original_probs = model.predict_proba(image)[0]
    predicted_class = model.classes_[np.argmax(original_probs)]
    class_col = int(np.flatnonzero(model.classes_ == predicted_class)[0])
    nblocks = 28 // patch_size
    variants = np.repeat(image, nblocks*nblocks, axis=0)
    for rr in range(nblocks):
        for cc in range(nblocks):
            pos = rr*nblocks+cc
            for r in range(rr*patch_size, (rr+1)*patch_size):
                start = r*28 + cc*patch_size
                variants[pos, start:start+patch_size] = 0
    new_probs = model.predict_proba(variants)[:, class_col].reshape(nblocks, nblocks)
    return predicted_class, original_probs[class_col], original_probs[class_col]-new_probs

local_idx = [np.flatnonzero(y_val == d)[0] for d in [1, 3, 5, 8]]
fig, axes = plt.subplots(4, 3, figsize=(11.6, 13))
for row, idx in enumerate(local_idx):
    pred, base_prob, dmap = local_occlusion(forest, X_val[idx], patch_size=4)
    heat = np.kron(dmap, np.ones((4, 4)))
    vmax = max(float(np.max(np.abs(heat))), 1e-9)
    original = X_val[idx].reshape(28,28)
    axes[row,0].imshow(original, cmap='gray', vmin=0, vmax=255)
    axes[row,0].set_title(f'Prawda {y_val[idx]} | prognoza {pred} | p={base_prob:.2f}')
    axes[row,1].imshow(heat, cmap='coolwarm', vmin=-vmax, vmax=vmax)
    axes[row,1].set_title('Lokalny spadek p po zasłonięciu')
    axes[row,2].imshow(original, cmap='gray', vmin=0, vmax=255)
    axes[row,2].imshow(heat, cmap='coolwarm', vmin=-vmax, vmax=vmax, alpha=.62)
    axes[row,2].set_title('Ślad decyzji na obrazie')
    for ax in axes[row]: ax.axis('off')
fig.suptitle('Globalna ważność ≠ lokalna wrażliwość pojedynczego obrazu', y=1.005)
plt.tight_layout(); savefig('16_lokalne_mapy_occlusion.png'); plt.show()


## 17 · Stabilność rankingu: czy inny las wskaże te same piksele? (opcjonalne)

`RUN_STABILITY=True` uruchamia dwa niezależne lasy na rozłącznych losowaniach próbek **ze zbioru treningowego**. Porównujemy rangę Spearmana i część wspólną TOP-50. Wyniki mogą się wahać: im mniejsza próba, tym bardziej.

In [ ]:
if RUN_STABILITY:
    stable_X, stable_y = stratified_cap(X_train, y_train,
                                       min(len(y_train), 8000), SEED+99)
    idx_a, idx_b = train_test_split(np.arange(len(stable_y)), test_size=.5,
                                   stratify=stable_y, random_state=SEED+99)
    imp_runs = []
    for ix, seed in [(idx_a,101), (idx_b,202)]:
        m = RandomForestClassifier(n_estimators=100, random_state=seed,
                                   n_jobs=N_JOBS)
        m.fit(stable_X[ix], stable_y[ix])
        imp_runs.append(m.feature_importances_)
    A, B = top_set(imp_runs[0]), top_set(imp_runs[1])
    print('Spearman:', spearmanr(imp_runs[0], imp_runs[1]).statistic)
    print('TOP-50 Jaccard:', len(A & B) / len(A | B))
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.7))
    show_heat(axes[0], imp_runs[0], 'Las A · osobna połowa treningu', cmap='hot')
    show_heat(axes[1], imp_runs[1], 'Las B · osobna połowa treningu', cmap='hot')
    plt.tight_layout(); savefig('16_stabilnosc.png'); plt.show()
else:
    print('Pominięto badanie stabilności. Ustaw RUN_STABILITY=True w konfiguracji.')

## 18 · Eksport wyników do CSV i obrazów PNG

Zapisujemy rankingi, wyniki porównań i zestawienia w folderze `mnist_piksele_wyniki`. Wszystkie ważne wykresy zapisują się na bieżąco do plików PNG. CSV można otworzyć w Excelu, R, pandas lub narzędziach BI.

In [ ]:
pixel_summary = pd.DataFrame({
    'piksel': np.arange(784),
    'wiersz': np.arange(784) // 28,
    'kolumna': np.arange(784) % 28,
    'mdi_random_forest': mdi,
    'mutual_information': mi,
    'wariancja': pixel_var,
    'waga_sgd_abs_srednia': abs_linear,
})
for d in range(10):
    pixel_summary[f'waga_sgd_cyfra_{d}'] = class_weights[d]
pixel_summary['ranga_mdi'] = pixel_summary['mdi_random_forest'].rank(ascending=False, method='min').astype(int)
pixel_summary['ranga_mi'] = pixel_summary['mutual_information'].rank(ascending=False, method='min').astype(int)
pixel_summary.sort_values('ranga_mdi').to_csv(OUT_DIR / 'ranking_pikseli.csv', index=False)
curve_df.to_csv(OUT_DIR / 'ablacja.csv', index=False)
selection_df.to_csv(OUT_DIR / 'wybor_liczby_pikseli.csv', index=False)
final_scores.to_csv(OUT_DIR / 'wyniki_test.csv', index=False)
pd.DataFrame(patch_drops).to_csv(OUT_DIR / 'permutacja_blokow.csv', index=False)

print('Wyniki zapisane w:', OUT_DIR.resolve())
for p in sorted(OUT_DIR.glob('*')):
    print(f'  {p.name:38} {p.stat().st_size / 1024:.1f} KB')

---
## 18 · Głębsze dno: pięć fundamentalnych rozróżnień

**1. Który piksel jest ważny, a dla kogo?** MDI ocenia piksele z perspektywy jednego lasu i wszystkich klas naraz. Wagi liniowe są **specyficzne dla klasy**. MI mierzy zależność piksela z etykietą, nie ważność dla konkretnego modelu.

**2. Korelacja nie jest przyczynowością.** Dwa piksele mogą współwystępować na tej samej kresce. Usunięcie jednego nie musi szkodzić modelowi, jeśli drugi zawiera podobną informację. Ważność nie jest równoznaczna z „powodem, dla którego człowiek widzi cyfrę 3”.

**3. Permutacja i wymazanie nie są tym samym.** Permutacja grupy zastępuje fragmenty obrazu fragmentami *z innych obrazów*, zachowując rozkład lokalnych wzorów, ale niszcząc powiązanie z etykietą. Ablacja wpisuje zera, więc może tworzyć nierealistyczne obrazy.

**4. Model z 100 pikselami a pełny model bez 100 pikseli rozwiązują inne problemy.** W pierwszym **uczymy nowy model** tylko na ograniczonej informacji. W drugim **psujemy dane wejściowe** modelowi nauczonemu na kompletnych obrazach. Wyników nie należy interpretować zamiennie.

**5. Co z generalizacją?** Jeśli wybieramy najlepszą liczbę cech na walidacji, test musi pozostać zamknięty do końca. Obserwowanie testu i poprawianie wyboru *k* na jego podstawie przecieka informację testową do procesu projektowania.

### Zestaw pytań na salę

1. Gdyby wszystkie cyfry były idealnie wyśrodkowane, czy ranking ważności wyglądałby inaczej?
2. Czy piksel może mieć wysoką MI i jednocześnie bardzo niską ważność MDI?
3. Dlaczego mapa dla cyfry **1** powinna różnić się od mapy dla **8**?
4. Jakie konsekwencje miałoby przesunięcie każdej cyfry o 3 piksele w prawo podczas testowania?
5. Co wyjaśniają negatywne wagi modelu liniowego, czego **nie** wyjaśnia MDI?
6. Czy możemy mówić o „istotnych pikselach” bez wskazania metody, modelu i rozkładu danych?

### Ćwiczenia rozszerzające (dla ambitnych)

- Zmień `FAST_MODE=False` i porównaj stabilność TOP-50.
- Przesuń obrazy w teście i sprawdź odporność modelu na przesunięcie; nie zmieniaj już pierwotnych wyników testowych w raporcie.
- Zbuduj **One-vs-Rest RandomForest** dla cyfr 3 i 8, aby otrzymać lasowe mapy specyficzne dla tych klas.
- Porównaj maskowanie zera z zastępowaniem średnią intensywnością piksela z treningu i opisz ograniczenia obu podejść.
- Zrób ocenę powtarzalności na 5 różnych `SEED` i przedstaw przedziały zmienności.

**Najważniejsza konkluzja:** interpretowalność modeli nie polega na „narysowaniu kolorowej mapy”, lecz na precyzyjnym określeniu *co* zostało zmierzone, *w jakich warunkach* i *jaką hipotezę* wynik wspiera.